In [183]:
import cv2
import pandas as pd
import numpy as np
import mediapipe as mp
import matplotlib.pyplot as plt
from matplotlib import patches
import os

# Inicializar MediaPipe
mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

In [184]:
# Controle de orientação do vídeo
ROTATE_MODE = 'auto_landscape'

# Controle de conexões (linhas)
CONNECTIONS_ENABLED = True

def _rotate_frame(frame, mode: str):
    if mode == 'none' or mode is None:
        return frame
    if mode == 'cw':
        return cv2.rotate(frame, cv2.ROTATE_90_CLOCKWISE)
    if mode == 'ccw':
        return cv2.rotate(frame, cv2.ROTATE_90_COUNTERCLOCKWISE)
    if mode == '180':
        return cv2.rotate(frame, cv2.ROTATE_180)
    if mode == 'auto_landscape':
        h, w = frame.shape[:2]
        if h > w:
            return cv2.rotate(frame, cv2.ROTATE_90_CLOCKWISE)
        return frame
    return frame

In [185]:
# Configurações das cores, marcadores e transparência por parte do corpo
LANDMARK_CONFIG = {
    'pose': {
        'color': (255, 0, 0),
        'marker': '.',
        'size': 300,
        'alpha': 1,
        'connections_color': (255, 255, 255)
    },
    'face': {
        'color': (57,255,20),
        'marker': '.',
        'size': 200,
        'alpha': 1,
    },
    'left_hand': {
        'color': (0, 0, 255),
        'marker': '.',
        'size': 250,
        'alpha': 1,
        'connections_color': (0, 0, 0)
    },
    'right_hand': {
        'color': (0, 0, 255),
        'marker': '.',
        'size': 250,
        'alpha': 1,
        'connections_color': (0, 0, 0)
    }
}

# Definir conexões do MediaPipe
POSE_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 7),
    (0, 4), (4, 5), (5, 6), (6, 8),
    (9, 10), (11, 12), (11, 13), (13, 15),
    (15, 17), (17, 19), (15, 19), (15, 21),
    (12, 14), (14, 16), (16, 18), (18, 20),
    (16, 20), (16, 22), (11, 23), (12, 24),
    (23, 24), (23, 25), (25, 27), (27, 29),
    (29, 31), (27, 31), (24, 26), (26, 28),
    (28, 30), (30, 32), (28, 32)
]

HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),     # Thumb
    (0, 5), (5, 6), (6, 7), (7, 8),     # Index
    (5, 9), (9, 10), (10, 11), (11, 12), # Middle
    (9, 13), (13, 14), (14, 15), (15, 16), # Ring
    (13, 17), (17, 18), (18, 19), (19, 20), # Pinky
    (0, 17)  # Palm
]


# groups = {
#     'lips': fm.FACEMESH_LIPS,
#     'left_eye': fm.FACEMESH_LEFT_EYE,
#     'right_eye': fm.FACEMESH_RIGHT_EYE,
#     'left_eyebrow': fm.FACEMESH_LEFT_EYEBROW,
#     'right_eyebrow': fm.FACEMESH_RIGHT_EYEBROW,
#     # Limites superior/inferior dos olhos (útil para contornos mais enxutos)
#     'left_eye_upper': fm.FACEMESH_LEFT_EYE_TOP_BOUNDARY,
#     'left_eye_lower': fm.FACEMESH_LEFT_EYE_BOTTOM_BOUNDARY,
#     'right_eye_upper': fm.FACEMESH_RIGHT_EYE_TOP_BOUNDARY,
#     'right_eye_lower': fm.FACEMESH_RIGHT_EYE_BOTTOM_BOUNDARY,
#     # Íris (se quiser)
#     'left_iris': getattr(fm, 'FACEMESH_LEFT_IRIS', set()),
#     'right_iris': getattr(fm, 'FACEMESH_RIGHT_IRIS', set()),
# }

### Orientação do vídeo

- Use `ROTATE_MODE` para manter todos os frames em horizontal (landscape) antes do overlay.
- Opções: `'none'`, `'cw'` (90° horário), `'ccw'` (90° anti-horário), `'180'`, `'auto_landscape'` (padrão: gira retratos para landscape).
- A rotação é aplicada ao frame lido do vídeo tanto no processamento em lote quanto no teste de um único frame.

### Conexões (linhas) entre pontos

- Use `CONNECTIONS_ENABLED` para ligar/desligar as linhas de conexão de pose e mãos.
- O rosto (face) não desenha conexões por design — apenas pontos.
- Valores: `True` (desenha linhas de pose/mãos) ou `False` (somente pontos em todas as partes).

All

In [186]:
# # Seleção de pontos a desenhar (edite conforme desejar)
# SELECTION = {
#     'pose': {
#         'enabled': True,
#     },
#     'left_hand': {
#         'enabled': True,
#     },
#     'right_hand': {
#         'enabled': True,
#     },
#     'face': {
#         'enabled': True,
#     }
# }

laines

In [187]:
# SELECTION = {
#     'pose': {
#         'enabled': True,
#         # Exemplo: apenas ombros e mãos da pose
#         'indices': [0, 11, 12, 13, 14, -1],
#     },
#     'left_hand': {
#         'enabled': True,
#     },
#     'right_hand': {
#         'enabled': True,
#     },
#     'face': {
#         'enabled': True,
#         # Use 'indices' para escolher índices absolutos OU 'groups' para regiões
#         'indices': [46, 52, 53, 65,
#                     295,283,282,276,
#                     7,159,155,145,
#                     382,386,249,374,
#                     324,13,78,14],
#         # 'groups': ['lips', 'left_eye', 'right_eye', 'left_eyebrow', 'right_eyebrow']
#                 #    'left_eye_upper', 'left_eye_lower', 'right_eye_upper', 'right_eye_lower',
#                 #    'left_iris', 'right_iris']
#     }
# }

arcanjo


In [188]:
# SELECTION = {
#     'pose': {
#         'enabled': True,
#     },
#     'left_hand': {
#         'enabled': True,
#     },
#     'right_hand': {
#         'enabled': True,
#     },
#     'face': {
#         'enabled': False,
#     }
# }

primeiro

In [189]:
# # Seleção de pontos a desenhar (edite conforme desejar)
# SELECTION = {
#     'pose': {
#         'enabled': False,
#         # Exemplo: apenas ombros e mãos da pose
#         'indices': [500, 502, 504, 501, 503, 505, 512, 513,
#                     513,505,503,501,
#                     512,504,502,500],
#     },
#     'left_hand': {
#         'enabled': True,
#         'indices': [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20],
#     },
#     'right_hand': {
#         'enabled': True,
#         'indices': [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20],
#     },
#     'face': {
#         'enabled': True,
#         'indices': [
#                     1,2,98,327,
#                     98,
#                     327,
#                     0,
#                     61, 185, 40, 39, 37, 267, 269, 270, 409,
#                     291, 146, 91, 181, 84, 17, 314, 405, 321, 375,
#                     78, 191, 80, 81, 82, 13, 312, 311, 310, 415,
#                     95, 88, 178, 87, 14, 317, 402, 318, 324, 308,
#                     84,181,91,146,61,185,40,39,37,87,178,88,95,78,191,80,81,82,
#                     84,181,91,146,61,185,40,39,37,87,178,88,95,78,191,80,81,82,

#                     33, 7, 163, 144, 145, 153, 154, 155, 133,
#                     246, 161, 160, 159, 158, 157, 173,
#                     263, 249, 390, 373, 374, 380, 381, 382, 362,
#                     466, 388, 387, 386, 385, 384, 398,
#                    ],
#     }
# }

segundo

In [190]:
# Seleção de pontos a desenhar (edite conforme desejar)
SELECTION = {
    'pose': {
        'enabled': True,
        'indices': [0, 1, 3, 4, 5, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23],
    },
    'left_hand': {
        'enabled': True,
    },
    'right_hand': {
        'enabled': True,
    },
    'face': {
        'enabled': True,
        'indices': [
            61, 185, 40, 39, 37, 0, 267, 269, 270,
            409, 291, 375, 321, 405, 314, 17, 84, 181
        ],
    }
}

In [191]:
def draw_landmarks_on_frame(frame, landmarks_row, min_visibility=0.5, connections: bool | None = None, selection: dict | None = None):
    """
    Desenha landmarks de MediaPipe em um frame do vídeo
    - Cores por parte: pose (branco), face (verde), mão esquerda (vermelho), mão direita (azul)
    - Marcadores por parte: pose(*), direita(/), esquerda(x), rosto(.)
    - Fonte pequena e sem legenda
    - Pose e mãos: conexões clipadas para não exceder a tela (opcional).
    - Suporta colunas com nomes (ex: hand_0_wrist_x) e índices (ex: pose_0_x).

    Parâmetros:
    - connections: True/False para desenhar ou esconder linhas. Se None, usa CONNECTIONS_ENABLED global.
    - selection: dicionário opcional para controlar quais pontos desenhar. Formatos suportados:
        {
          'pose': {
             'enabled': True|False,               # habilita/desabilita pose
             'indices': [0, 11, 12, ...]         # desenha apenas esses índices (MediaPipe Pose)
          },
          'left_hand': {
             'enabled': True|False,
             'indices': [0, 4, 8, ...]           # índices da mão (0..20)
          },
          'right_hand': {...},
          'face': {
             'enabled': True|False,
             'indices': [13, 14, ...],           # índices absolutos do Face Mesh (0..468/478)
             'groups': ['lips', 'left_eye', 'right_eye', 'left_eyebrow', 'right_eyebrow',
                        'left_eye_upper', 'left_eye_lower', 'right_eye_upper', 'right_eye_lower',
                        'left_iris', 'right_iris']  # união dos grupos solicitados
          }
        }
    """
    # Configurações do plot
    fig = plt.figure(figsize=(12, 8), facecolor='black')
    plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB), alpha=0.8)


    height, width = frame.shape[:2]

    def in_frame(px, py, w, h, margin=0):
        return (margin <= px < w - margin) and (margin <= py < h - margin)

    # Cohen–Sutherland line clipping
    INSIDE, LEFT, RIGHT, BOTTOM, TOP = 0, 1, 2, 4, 8

    def _code(x, y, xmin, ymin, xmax, ymax):
        code = INSIDE
        if x < xmin:
            code |= LEFT
        elif x > xmax:
            code |= RIGHT
        if y < ymin:
            code |= BOTTOM
        elif y > ymax:
            code |= TOP
        return code

    def clip_line(x1, y1, x2, y2, xmin, ymin, xmax, ymax):
        code1 = _code(x1, y1, xmin, ymin, xmax, ymax)
        code2 = _code(x2, y2, xmin, ymin, xmax, ymax)

        accept = False
        while True:
            if not (code1 | code2):
                accept = True
                break
            elif code1 & code2:
                break
            else:
                out_code = code1 if code1 else code2
                if out_code & TOP:
                    x = x1 + (x2 - x1) * (height - 1 - y1) / (y2 - y1) if (y2 - y1) != 0 else x1
                    y = height - 1
                elif out_code & BOTTOM:
                    x = x1 + (x2 - x1) * (0 - y1) / (y2 - y1) if (y2 - y1) != 0 else x1
                    y = 0
                elif out_code & RIGHT:
                    y = y1 + (y2 - y1) * (width - 1 - x1) / (x2 - x1) if (x2 - x1) != 0 else y1
                    x = width - 1
                elif out_code & LEFT:
                    y = y1 + (y2 - y1) * (0 - x1) / (x2 - x1) if (x2 - x1) != 0 else y1
                    x = 0
                if out_code == code1:
                    x1, y1 = x, y
                    code1 = _code(x1, y1, xmin, ymin, xmax, ymax)
                else:
                    x2, y2 = x, y
                    code2 = _code(x2, y2, xmin, ymin, xmax, ymax)
        if accept:
            return x1, y1, x2, y2
        else:
            return None

    # Mapas de nome -> índice seguindo MediaPipe
    pose_name_to_idx = {lm.name.lower(): lm.value for lm in mp.solutions.pose.PoseLandmark}
    hand_name_to_idx = {lm.name.lower(): lm.value for lm in mp.solutions.hands.HandLandmark}

    # Utilitário para pegar valor do row com segurança
    def get_val(prefix, key, axis):
        col = f"{prefix}_{key}_{axis}"
        return landmarks_row.get(col, np.nan)

    # Extrair POSE: aceitar nomeado (pose_left_shoulder_x) ou indexado (pose_11_x)
    def extract_pose_landmarks(row):
        pts = {}
        # Tentar por nomes conhecidos
        for name, idx in pose_name_to_idx.items():
            x, y, z = get_val('pose', name, 'x'), get_val('pose', name, 'y'), get_val('pose', name, 'z')
            if pd.notna(x) and pd.notna(y):
                px, py = float(x) * width, float(y) * height
                pts[idx] = (px, py, z)
        if pts:
            return pts
        # Fallback por índices sequenciais
        i = 0
        while f'pose_{i}_x' in row.index:
            x, y, z = row.get(f'pose_{i}_x', np.nan), row.get(f'pose_{i}_y', np.nan), row.get(f'pose_{i}_z', np.nan)
            if pd.notna(x) and pd.notna(y):
                px, py = float(x) * width, float(y) * height
                pts[i] = (px, py, z)
            i += 1
        return pts

    # Extrair MÃOS: aceitar nomeado (hand_0_wrist_x) e produzir dict idx->(x,y,z)
    def extract_hand_map(row, hand_id):
        pts = {}
        prefix = f'hand_{hand_id}'
        # Tentar por nomes conhecidos
        for name, idx in hand_name_to_idx.items():
            x, y, z = get_val(prefix, name, 'x'), get_val(prefix, name, 'y'), get_val(prefix, name, 'z')
            if pd.notna(x) and pd.notna(y):
                px, py = float(x) * width, float(y) * height
                # Ignorar zeros absolutos (caso placeholder 0 do extrator)
                if not (px == 0 and py == 0):
                    pts[idx] = (px, py, z)
        if pts:
            return pts
        # Fallback por índices sequenciais (pouco provável nesse dataset)
        i = 0
        while f'{prefix}_{i}_x' in row.index:
            x, y, z = row.get(f'{prefix}_{i}_x', np.nan), row.get(f'{prefix}_{i}_y', np.nan), row.get(f'{prefix}_{i}_z', np.nan)
            if pd.notna(x) and pd.notna(y):
                px, py = float(x) * width, float(y) * height
                if not (px == 0 and py == 0):
                    pts[i] = (px, py, z)
            i += 1
        return pts

    # Extrair ROSTO como dict idx->(x,y,z)
    def extract_face_landmarks(row):
        pts = {}
        i = 0
        while f'face_{i}_x' in row.index:
            x, y, z = row.get(f'face_{i}_x', np.nan), row.get(f'face_{i}_y', np.nan), row.get(f'face_{i}_z', np.nan)
            if pd.notna(x) and pd.notna(y):
                px, py = float(x) * width, float(y) * height
                if not (px == 0 and py == 0):
                    pts[i] = (px, py, z)
            i += 1
        return pts

    # Extrair todos
    pose_map = extract_pose_landmarks(landmarks_row)
    left_map = extract_hand_map(landmarks_row, 0)
    right_map = extract_hand_map(landmarks_row, 1)
    face_map = extract_face_landmarks(landmarks_row)

    draw_connections = CONNECTIONS_ENABLED if connections is None else connections

    # Preparar filtro de seleção
    def is_enabled(part: str, default=True):
        if selection is None:
            return default
        cfg = selection.get(part)
        if cfg is None:
            return default
        if isinstance(cfg, dict):
            return cfg.get('enabled', default)
        if isinstance(cfg, bool):
            return cfg
        return default

    def selected_indices(part: str, total_keys: set[int]) -> set[int]:
        if selection is None:
            return set(total_keys)
        cfg = selection.get(part)
        if cfg is None:
            return set(total_keys)
        idxs: set[int] = set()
        # índices explícitos
        if isinstance(cfg, dict) and 'indices' in cfg and cfg['indices'] is not None:
            idxs.update(int(i) for i in cfg['indices'])
        # grupos da face
        if part == 'face' and isinstance(cfg, dict) and cfg.get('groups'):
            fm = mp.solutions.face_mesh
            groups_raw = {
                'lips': getattr(fm, 'FACEMESH_LIPS', set()),
                'left_eye': getattr(fm, 'FACEMESH_LEFT_EYE', set()),
                'right_eye': getattr(fm, 'FACEMESH_RIGHT_EYE', set()),
                'left_eyebrow': getattr(fm, 'FACEMESH_LEFT_EYEBROW', set()),
                'right_eyebrow': getattr(fm, 'FACEMESH_RIGHT_EYEBROW', set()),
                'left_eye_upper': getattr(fm, 'FACEMESH_LEFT_EYE_TOP_BOUNDARY', set()),
                'left_eye_lower': getattr(fm, 'FACEMESH_LEFT_EYE_BOTTOM_BOUNDARY', set()),
                'right_eye_upper': getattr(fm, 'FACEMESH_RIGHT_EYE_TOP_BOUNDARY', set()),
                'right_eye_lower': getattr(fm, 'FACEMESH_RIGHT_EYE_BOTTOM_BOUNDARY', set()),
                'left_iris': getattr(fm, 'FACEMESH_LEFT_IRIS', set()),
                'right_iris': getattr(fm, 'FACEMESH_RIGHT_IRIS', set()),
            }
            for g in cfg['groups']:
                conns = groups_raw.get(g, [])
                for a, b in conns:
                    idxs.add(int(a)); idxs.add(int(b))
        # Se nada especificado, usa todos
        if not idxs:
            return set(total_keys)
        # Interseção com os que existem
        return set(total_keys).intersection(idxs)

    # Aplicar filtros
    if is_enabled('pose'):
        pose_keys = set(pose_map.keys())
        pose_sel = selected_indices('pose', pose_keys)
        pose_map = {i: v for i, v in pose_map.items() if i in pose_sel}
    else:
        pose_map = {}

    if is_enabled('left_hand'):
        l_keys = set(left_map.keys())
        l_sel = selected_indices('left_hand', l_keys)
        left_map = {i: v for i, v in left_map.items() if i in l_sel}
    else:
        left_map = {}

    if is_enabled('right_hand'):
        r_keys = set(right_map.keys())
        r_sel = selected_indices('right_hand', r_keys)
        right_map = {i: v for i, v in right_map.items() if i in r_sel}
    else:
        right_map = {}

    if is_enabled('face'):
        f_keys = set(face_map.keys())
        f_sel = selected_indices('face', f_keys)
        face_map = {i: v for i, v in face_map.items() if i in f_sel}
    else:
        face_map = {}

    # Desenhar POSE (pontos dentro da tela; conexões clipadas)
    if pose_map:
        config = LANDMARK_CONFIG['pose']
        pts_in = [(i, v) for i, v in pose_map.items() if in_frame(v[0], v[1], width, height, margin=0)]
        if pts_in:
            x_coords, y_coords = zip(*[(lm[1][0], lm[1][1]) for lm in pts_in])
            plt.scatter(x_coords, y_coords,
                        c=[np.array(config['color'])/255.0],
                        marker=config['marker'],
                        s=config['size'],
                        alpha=config['alpha'],
                        edgecolors='white',
                        linewidth=2,
                        zorder=3)

        if draw_connections:
            xmin, ymin, xmax, ymax = 0, 0, width - 1, height - 1
            for a, b in POSE_CONNECTIONS:
                if a in pose_map and b in pose_map:
                    x1, y1 = pose_map[a][0], pose_map[a][1]
                    x2, y2 = pose_map[b][0], pose_map[b][1]
                    clipped = clip_line(x1, y1, x2, y2, xmin, ymin, xmax, ymax)
                    if clipped is not None:
                        cx1, cy1, cx2, cy2 = clipped
                        plt.plot([cx1, cx2], [cy1, cy2],
                                 color=np.array(config['connections_color'])/255.0,
                                 linewidth=2,
                                 alpha=config['alpha'],
                                 zorder=2)

    # Desenhar ROSTO
    if face_map:
        config = LANDMARK_CONFIG['face']
        x_coords, y_coords = zip(*[(pt[0], pt[1]) for pt in face_map.values()])
        plt.scatter(x_coords, y_coords,
                    c=[np.array(config['color'])/255.0],
                    marker=config['marker'],
                    s=config['size'],
                    alpha=config['alpha'],
                    edgecolors='black',
                    linewidth=2,
                    zorder=1)

    # Função para desenhar MÃO genérica por mapa (dict idx->pt)
    def draw_hand(map_pts, config):
        if not map_pts:
            return
        if draw_connections:
            xmin, ymin, xmax, ymax = 0, 0, width - 1, height - 1
            # Conexões clipadas
            for u, v in HAND_CONNECTIONS:
                if u in map_pts and v in map_pts:
                    su = map_pts[u]
                    ev = map_pts[v]
                    clipped = clip_line(su[0], su[1], ev[0], ev[1], xmin, ymin, xmax, ymax)
                    if clipped is not None:
                        cx1, cy1, cx2, cy2 = clipped
                        plt.plot([cx1, cx2], [cy1, cy2],
                                 color=np.array(config['connections_color'])/255.0,
                                 linewidth=2.0,
                                 alpha=config['alpha'],
                                 zorder=1)
        # Pontos com scatter
        xs = [pt[0] for pt in map_pts.values()]
        ys = [pt[1] for pt in map_pts.values()]
        if xs and ys:
            plt.scatter(xs, ys,
                        c=[np.array(config['color'])/255.0],
                        marker=config.get('marker', '.'),
                        s=config.get('size', 40),
                        alpha=config.get('alpha', 1),
                        edgecolors='black',
                        linewidth=2.0,
                        zorder=3)

    # Desenhar MÃOS
    draw_hand(left_map, LANDMARK_CONFIG['left_hand'])
    draw_hand(right_map, LANDMARK_CONFIG['right_hand'])

    # Finalizar
    plt.axis('off')
    plt.gca().set_aspect('equal', adjustable='box')
    plt.tight_layout(pad=0)

    return fig

In [192]:
# Carregar o vídeo e dados de landmarks
video_path = 'palmas.mp4'
landmarks_file = 'landmarks_full.csv'  # ou 'landmarks [2].csv'

# Verificar se os arquivos existem
if not os.path.exists(video_path):
    print(f"Arquivo de vídeo não encontrado: {video_path}")
if not os.path.exists(landmarks_file):
    print(f"Arquivo de landmarks não encontrado: {landmarks_file}")

# Carregar landmarks
try:
    landmarks_df = pd.read_csv(landmarks_file)
    print(f"Landmarks carregados: {len(landmarks_df)} registros")
    print(f"Colunas disponíveis: {landmarks_df.columns.tolist()}")
    print(f"Frames únicos: {landmarks_df['frame'].nunique() if 'frame' in landmarks_df.columns else 'Coluna frame não encontrada'}")
except Exception as e:
    print(f"Erro ao carregar landmarks: {e}")

# Verificar estrutura dos dados
if 'landmarks_df' in locals():
    print("\nPrimeiras linhas dos dados:")
    print(landmarks_df.head())

Landmarks carregados: 194 registros
Colunas disponíveis: ['category', 'video_name', 'frame', 'person', 'video_path', 'dataset', 'extractor', 'hand_0_0_x', 'hand_0_0_y', 'hand_0_0_z', 'hand_0_1_x', 'hand_0_1_y', 'hand_0_1_z', 'hand_0_2_x', 'hand_0_2_y', 'hand_0_2_z', 'hand_0_3_x', 'hand_0_3_y', 'hand_0_3_z', 'hand_0_4_x', 'hand_0_4_y', 'hand_0_4_z', 'hand_0_5_x', 'hand_0_5_y', 'hand_0_5_z', 'hand_0_6_x', 'hand_0_6_y', 'hand_0_6_z', 'hand_0_7_x', 'hand_0_7_y', 'hand_0_7_z', 'hand_0_8_x', 'hand_0_8_y', 'hand_0_8_z', 'hand_0_9_x', 'hand_0_9_y', 'hand_0_9_z', 'hand_0_10_x', 'hand_0_10_y', 'hand_0_10_z', 'hand_0_11_x', 'hand_0_11_y', 'hand_0_11_z', 'hand_0_12_x', 'hand_0_12_y', 'hand_0_12_z', 'hand_0_13_x', 'hand_0_13_y', 'hand_0_13_z', 'hand_0_14_x', 'hand_0_14_y', 'hand_0_14_z', 'hand_0_15_x', 'hand_0_15_y', 'hand_0_15_z', 'hand_0_16_x', 'hand_0_16_y', 'hand_0_16_z', 'hand_0_17_x', 'hand_0_17_y', 'hand_0_17_z', 'hand_0_18_x', 'hand_0_18_y', 'hand_0_18_z', 'hand_0_19_x', 'hand_0_19_y', 'han

In [193]:
# Abrir vídeo e processar frames
cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    raise RuntimeError(f"Não foi possível abrir o vídeo: {video_path}")

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

print(f"Total de frames no vídeo: {total_frames}")
print(f"FPS: {fps}")

# Criar diretório para salvar as imagens
output_dir = 'frames_com_landmarks'
os.makedirs(output_dir, exist_ok=True)

# Configurar quantos frames processar (ajuste conforme necessário)
frame_step = 2  # Processar a cada 2 frames (use 1 para salvar todos)
max_frames = None  # None = processar até o fim do vídeo; ou defina um número (ex: 1000)
end_frame = total_frames if max_frames is None else min(total_frames, max_frames)
frames_to_process = list(range(0, end_frame, frame_step))

print(
    f"Processando {len(frames_to_process)} frames (step={frame_step}, "
    f"limite={'todos' if max_frames is None else end_frame})..."
)

saved_count = 0

for frame_idx in frames_to_process:
    # Ir para o frame específico
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ret, frame = cap.read()

    if not ret:
        print(f"Erro ao ler frame {frame_idx}")
        continue

    # Rotacionar conforme configuração
    frame = _rotate_frame(frame, ROTATE_MODE)

    try:
        # Selecionar a linha de landmarks do frame (assumindo coluna 'frame')
        if 'frame' in landmarks_df.columns:
            rows = landmarks_df[landmarks_df['frame'] == frame_idx]
            if rows.empty:
                print(f"Sem landmarks para frame {frame_idx}")
                continue
            row = rows.iloc[0]
        else:
            # Se não houver coluna 'frame', assumir uma linha por frame na ordem
            if frame_idx >= len(landmarks_df):
                print(f"Sem linha de landmarks para índice {frame_idx}")
                continue
            row = landmarks_df.iloc[frame_idx]

        # Desenhar landmarks no frame
        fig = draw_landmarks_on_frame(frame, row, selection=SELECTION)

        # Salvar como PNG
        output_path = os.path.join(output_dir, f'frame_{frame_idx:04d}_landmarks.png')
        fig.savefig(output_path, dpi=150, bbox_inches='tight', pad_inches=0.0, facecolor='black')
        plt.close(fig)  # Fechar figura para liberar memória

        saved_count += 1
        if saved_count % 5 == 0:
            print(f"Processados {saved_count} frames...")

    except Exception as e:
        print(f"Erro ao processar frame {frame_idx}: {e}")

cap.release()
print(f"\nProcessamento concluído! {saved_count} frames salvos em '{output_dir}'")

Total de frames no vídeo: 194
FPS: 60
Processando 97 frames (step=2, limite=todos)...
Processados 5 frames...
Processados 10 frames...
Processados 15 frames...
Processados 20 frames...
Processados 25 frames...
Processados 30 frames...
Processados 35 frames...
Processados 40 frames...
Processados 45 frames...
Processados 50 frames...
Processados 55 frames...
Processados 60 frames...
Processados 65 frames...
Processados 70 frames...
Processados 75 frames...
Processados 80 frames...
Processados 85 frames...
Processados 90 frames...
Processados 95 frames...

Processamento concluído! 97 frames salvos em 'frames_com_landmarks'


In [194]:
# Teste com um único frame (robusto)
print("Testando função com frame 0...")

# Fechar captura anterior (se existir)
try:
    if 'cap' in globals() and isinstance(cap, cv2.VideoCapture):
        cap.release()
except Exception:
    pass

# Reabrir o vídeo com checagem
cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    print(f"Não foi possível abrir o vídeo: {video_path}")
else:
    # Pegar o primeiro frame
    ret, test_frame = cap.read()

    if ret and test_frame is not None:
        print(f"Frame carregado (antes da rotação): {test_frame.shape}")
        test_frame = _rotate_frame(test_frame, ROTATE_MODE)
        print(f"Frame após rotação: {test_frame.shape}")

        # Pegar landmarks correspondentes
        first_row = None
        try:
            if isinstance(landmarks_df, pd.DataFrame) and len(landmarks_df) > 0:
                if 'frame' in landmarks_df.columns:
                    rows = landmarks_df[landmarks_df['frame'] == 0]
                    if not rows.empty:
                        first_row = rows.iloc[0]
                    else:
                        print("Sem landmarks para frame 0; usando primeira linha disponível do CSV.")
                        first_row = landmarks_df.iloc[0]
                else:
                    first_row = landmarks_df.iloc[0]
            else:
                print("DataFrame de landmarks vazio ou inválido.")
        except Exception as e:
            print(f"Falha ao selecionar linha de landmarks: {e}")

        if first_row is not None:
            # Verificar quantos landmarks temos de cada tipo (garante que o índice é string)
            idx_iter = [c for c in first_row.index if isinstance(c, str)]
            hand_cols = [col for col in idx_iter if col.startswith('hand_') and col.endswith('_x')]
            face_cols = [col for col in idx_iter if col.startswith('face_') and col.endswith('_x')]
            pose_cols = [col for col in idx_iter if col.startswith('pose_') and col.endswith('_x')]

            print("Landmarks encontrados:")
            print(f"- Hand: {len(hand_cols)} pontos")
            print(f"- Face: {len(face_cols)} pontos")
            print(f"- Pose: {len(pose_cols)} pontos")

            try:
                # Desenhar landmarks com seleção
                fig = draw_landmarks_on_frame(test_frame, first_row, selection=SELECTION)

                # Salvar como PNG (removido edgecolor para compatibilidade)
                output_filename = "improved_test_frame.png"
                fig.savefig(
                    output_filename,
                    dpi=150,
                    bbox_inches='tight',
                    pad_inches=0.0,
                    facecolor='black'
                )
                plt.close(fig)
                print(f"Frame salvo como: {output_filename}")
            except Exception as e:
                print(f"Erro ao desenhar/salvar o frame: {e}")
        else:
            print("Não foi possível obter uma linha de landmarks válida.")
    else:
        print("Erro ao carregar frame 0 do vídeo.")

# Liberar captura atual
try:
    if 'cap' in globals() and isinstance(cap, cv2.VideoCapture):
        cap.release()
except Exception:
    pass

Testando função com frame 0...
Frame carregado (antes da rotação): (1080, 1440, 3)
Frame após rotação: (1080, 1440, 3)
Landmarks encontrados:
- Hand: 42 pontos
- Face: 468 pontos
- Pose: 33 pontos
Frame salvo como: improved_test_frame.png


In [195]:
# Verificar estrutura dos dados de landmarks
print("Estrutura do DataFrame landmarks:")
print(f"Shape: {landmarks_df.shape}")
print(f"Colunas: {landmarks_df.columns.tolist()}")
print("\nPrimeiras 5 linhas:")
print(landmarks_df.head())
print("\nTipos de dados:")
print(landmarks_df.dtypes)
print("\nValores únicos em algumas colunas importantes:")
if 'frame' in landmarks_df.columns:
    print(f"Frames únicos: {sorted(landmarks_df['frame'].unique())[:10]}")  # Primeiros 10
if 'type' in landmarks_df.columns:
    print(f"Tipos de landmarks: {landmarks_df['type'].unique()}")
else:
    print("Coluna 'type' não encontrada. Verificando outras colunas...")

Estrutura do DataFrame landmarks:
Shape: (194, 1636)
Colunas: ['category', 'video_name', 'frame', 'person', 'video_path', 'dataset', 'extractor', 'hand_0_0_x', 'hand_0_0_y', 'hand_0_0_z', 'hand_0_1_x', 'hand_0_1_y', 'hand_0_1_z', 'hand_0_2_x', 'hand_0_2_y', 'hand_0_2_z', 'hand_0_3_x', 'hand_0_3_y', 'hand_0_3_z', 'hand_0_4_x', 'hand_0_4_y', 'hand_0_4_z', 'hand_0_5_x', 'hand_0_5_y', 'hand_0_5_z', 'hand_0_6_x', 'hand_0_6_y', 'hand_0_6_z', 'hand_0_7_x', 'hand_0_7_y', 'hand_0_7_z', 'hand_0_8_x', 'hand_0_8_y', 'hand_0_8_z', 'hand_0_9_x', 'hand_0_9_y', 'hand_0_9_z', 'hand_0_10_x', 'hand_0_10_y', 'hand_0_10_z', 'hand_0_11_x', 'hand_0_11_y', 'hand_0_11_z', 'hand_0_12_x', 'hand_0_12_y', 'hand_0_12_z', 'hand_0_13_x', 'hand_0_13_y', 'hand_0_13_z', 'hand_0_14_x', 'hand_0_14_y', 'hand_0_14_z', 'hand_0_15_x', 'hand_0_15_y', 'hand_0_15_z', 'hand_0_16_x', 'hand_0_16_y', 'hand_0_16_z', 'hand_0_17_x', 'hand_0_17_y', 'hand_0_17_z', 'hand_0_18_x', 'hand_0_18_y', 'hand_0_18_z', 'hand_0_19_x', 'hand_0_19_y',